In [1]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 8.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 36.4 MB/s eta 0:00:00:00:010:01m


In [7]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import PeftModel
from huggingface_hub import hf_hub_download
import shutil



In [4]:
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_PATH = "/kaggle/input/models/koushikikundu/mistral-hr/pytorch/default/1"


print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

model.eval()

print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loading LoRA adapter...
Model loaded successfully!


In [5]:
# Merge LoRA adapter into the base model

print("Merging LoRA adapter...")

merged_model = model.merge_and_unload()

print("LoRA merged successfully!")

Merging LoRA adapter...
LoRA merged successfully!


In [6]:
MERGED_PATH = "/kaggle/working/mistral_hr_merged"

print("Saving merged model...")

merged_model.save_pretrained(
    MERGED_PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(MERGED_PATH)

print("Saved to:", MERGED_PATH)

Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /kaggle/working/mistral_hr_merged


In [8]:

tokenizer_model = hf_hub_download(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    filename="tokenizer.model"
)

shutil.copy(
    tokenizer_model,
    "/kaggle/working/mistral_hr_merged/tokenizer.model"
)

print("Copied tokenizer.model")

Copied tokenizer.model


In [9]:
!cp -r /kaggle/working/mistral_hr_merged /tmp/mistral_hr_merged

In [10]:
!ls -lh /tmp/mistral_hr_merged

total 14G
-rw-r--r-- 1 root root 3.9K Aug 27 05:18 chat_template.jinja
-rw-r--r-- 1 root root  688 Aug 27 05:17 config.json
-rw-r--r-- 1 root root  110 Aug 27 05:17 generation_config.json
-rw-r--r-- 1 root root  14G Aug 27 05:18 model.safetensors
-rw-r--r-- 1 root root  431 Aug 27 05:18 tokenizer_config.json
-rw-r--r-- 1 root root 3.6M Aug 27 05:18 tokenizer.json
-rw-r--r-- 1 root root 574K Aug 27 05:18 tokenizer.model


In [11]:
!git clone https://github.com/ggml-org/llama.cpp.git

Cloning into 'llama.cpp'...
remote: Enumerating objects: 117641, done.
remote: Counting objects: 100% (480/480), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 117641 (delta 358), reused 295 (delta 283), pack-reused 117161 (from 3)
Receiving objects: 100% (117641/117641), 423.88 MiB | 27.50 MiB/s, done.
Resolving deltas: 100% (82638/82638), done.


In [12]:
!pip install -q -r llama.cpp/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 893.1 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 8.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [13]:
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /tmp/mistral_hr_merged \
    --outfile /tmp/mistral_hr_f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: mistral_hr_merged
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {4096, 32768}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 32768}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {4096, 1024}
INFO:hf-to-gguf:blk.0.att

In [14]:
!ls -lh /tmp/mistral_hr_f16.gguf

-rw-r--r-- 1 root root 14G Aug 27 05:21 /tmp/mistral_hr_f16.gguf


In [15]:
%cd /kaggle/working/llama.cpp
!cmake -B build
!cmake --build build --config Release -j 4

/kaggle/working/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.3.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYS

In [16]:
!find /kaggle/working/llama.cpp/build -type f -name "llama-quantize"

/kaggle/working/llama.cpp/build/bin/llama-quantize


In [17]:
! /kaggle/working/llama.cpp/build/bin/llama-quantize \
    /tmp/mistral_hr_f16.gguf \
    /tmp/mistral_hr_Q4_K_M.gguf \
    Q4_K_M

version: 0.3.0-dev (build 10644, commit d7a207411)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/tmp/mistral_hr_f16.gguf' to '/tmp/mistral_hr_Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from /tmp/mistral_hr_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Mistral_Hr_Merged
llama_model_loader: - kv   3:                         general.size_label str              = 7.2B
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                       llama.context_length u32       

In [18]:
!ls -lh /tmp/mistral_hr_Q4_K_M.gguf

-rw-r--r-- 1 root root 4.1G Aug 27 05:37 /tmp/mistral_hr_Q4_K_M.gguf


In [19]:
!cp /tmp/mistral_hr_Q4_K_M.gguf /kaggle/working/

In [20]:
!ls -lh /kaggle/working/mistral_hr_Q4_K_M.gguf

-rw-r--r-- 1 root root 4.1G Aug 27 05:39 /kaggle/working/mistral_hr_Q4_K_M.gguf


In [21]:
!mkdir -p /tmp/hr_gguf_dataset

In [22]:
!cp /tmp/mistral_hr_Q4_K_M.gguf /tmp/hr_gguf_dataset/

In [23]:
!ls -lh /tmp/hr_gguf_dataset/

total 4.1G
-rw-r--r-- 1 root root 4.1G Aug 27 05:39 mistral_hr_Q4_K_M.gguf


In [24]:
GGUF_PATH = "/tmp/mistral_hr_Q4_K_M.gguf"
UPLOAD_DIR = "/tmp/mistral_hr_ollama"

os.makedirs(UPLOAD_DIR, exist_ok=True)

shutil.copy2(
    GGUF_PATH,
    os.path.join(UPLOAD_DIR, "mistral_hr_Q4_K_M.gguf")
)

print(os.listdir(UPLOAD_DIR))

['mistral_hr_Q4_K_M.gguf']


In [25]:
path = "/tmp/mistral_hr_ollama/mistral_hr_Q4_K_M.gguf"

print(
    f"{os.path.getsize(path) / (1024**3):.2f} GB"
)

4.07 GB


In [26]:
import kagglehub

kagglehub.model_upload(
    "koushikikundu/mistral-hr-gguf/gguf/default",
    "/tmp/mistral_hr_ollama",
    license_name="apache-2.0",
    version_notes="Q4_K_M GGUF fine-tuned Mistral HR model"
)

Uploading Model https://api.kaggle.com/models/koushikikundu/mistral-hr-gguf/gguf/default ...


Uploading: 100%|██████████| 4.37G/4.37G [03:08<00:00, 23.2MB/s]  

Upload successful: /tmp/mistral_hr_ollama/mistral_hr_Q4_K_M.gguf (4GB)


Your model instance version has been created.
Files are being processed...
See at: https://api.kaggle.com/models/koushikikundu/mistral-hr-gguf/gguf/default
